# 09 - SOTA Comparison
Compare our methods against state-of-the-art baselines:
- Trimesh fill-all (traditional generic hole filling)
- Advancing-front + RPA (Liepa-style)
- Open3D Poisson reconstruction (global completion)

Also evaluates Chamfer Distance and Hausdorff Distance
against the complete (ground-truth) mesh.

**Prerequisite**: Run notebook 01 first to build the dataset.

In [1]:
import sys, os, gc
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import load_config, ensure_dirs
from src.data.dataset_index import DatasetIndex
from src.experiments.run_batch import run_batch_experiment
from src.experiments.summarize import summarize_results
from src.visualization.plot_utils import (
    plot_quantitative_summary, plot_sota_comparison,
    plot_failure_cases, set_paper_style,
)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'chair_leg.yaml'))
ensure_dirs(cfg)
index = DatasetIndex(cfg['paths']['raw_data_dir'])
print(f'Dataset: {len(index)} samples')

Dataset: 100 samples


In [3]:
# Run FULL experiment including SOTA and distance metrics
# This may take 10-30 minutes depending on dataset size
df = run_batch_experiment(
    data_dir=cfg['paths']['raw_data_dir'],
    output_dir=cfg['paths']['output_dir'],
    margin=cfg['repair']['margin'],
    proximity_threshold=cfg['repair']['proximity_threshold'],
    save_meshes=True,
    include_distance=True,
    include_sota=True,
)
print(f'Results: {len(df)} samples')
df.head()

2026-04-13 22:27:57 [INFO] BatchExperiment: Running on 100 samples (SOTA=True, distance=True)
Running experiments: 100%|█████████████████████████████████████████████████████████| 100/100 [1:56:13<00:00, 69.73s/it]
2026-04-14 00:24:10 [INFO] BatchExperiment: Done: 84 success, 16 failed


Results: 84 samples


,sample_id,n_boundary_loops,n_target_loops_rpa,center_fan/target_loop_length_before,center_fan/target_loop_length_after,center_fan/improvement,center_fan/closure_ratio,center_fan/n_new_vertices,center_fan/n_new_faces,center_fan/avg_new_face_quality,...,advancing_front_rpa/std_new_face_quality,advancing_front_rpa/n_faces_inside,advancing_front_rpa/n_faces_outside,advancing_front_rpa/locality_ratio,advancing_front_rpa/chamfer_distance,advancing_front_rpa/hausdorff_distance,advancing_front_rpa/dev_mean,advancing_front_rpa/dev_max,advancing_front_rpa/dev_median,advancing_front_rpa/dev_std
0,2882,128,40,13.685422,0.0,13.685422,1.0,40.0,1488.0,0.289861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40584,24,18,4.972269,0.0,4.972269,1.0,18.0,876.0,0.254221,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,42201,32,32,19.469631,0.0,19.469631,1.0,32.0,1607.0,0.198347,...,0.122496,1543.0,0.0,1.0,0.109441,0.671726,0.010202,0.035770,0.009829,0.004972
3,41898,32,24,2.674532,0.0,2.674532,1.0,24.0,534.0,0.530598,...,0.199315,486.0,0.0,1.0,0.028065,0.346116,0.011292,0.042294,0.010457,0.006152
4,39573,163,107,68.148484,0.0,68.148484,1.0,107.0,3826.0,0.266503,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Generate all tables including SOTA
tables = summarize_results(df, cfg['paths']['output_dir'])
tables["table6_sota_comparison"]

,Method,Residual ↓,Improvement ↑,New Vtx ↓,New Faces ↓,Avg Quality ↑,Locality ↑,CD ↓,HD ↓
0,Center-fan,0.000000,39.951671,43.750000,1761.369048,0.267057,0.665615,0.081629,0.602517
1,Planar + RPA,1.938363,38.013307,0.000000,1656.940476,0.434137,0.671032,0.081144,0.602516
2,Planar + LH-only,38.256894,1.694776,0.000000,96.190476,0.412174,0.571698,0.079868,0.604043
3,Trimesh fill-all,35.813887,4.137784,0.000000,66.702381,0.119059,0.065686,0.079017,0.603997
4,Adv-front + RPA,0.000000,26.248969,0.000000,1425.604167,0.440602,0.717385,0.079703,0.589082
5,Poisson recon,2.891222,37.060449,1400.238095,2403.035714,0.000000,0.000000,0.097364,0.602787


In [5]:
# Identify which SOTA methods actually ran successfully
available_methods = []
for m in ['center_fan', 'planar_removed_part_aware', 'planar_largest_hole_only',
          'trimesh_fill_all', 'advancing_front_rpa', 'open3d_poisson']:
    col = f'{m}/target_loop_length_after'
    if col in df.columns and df[col].notna().sum() > 0:
        available_methods.append(m)
        print(f'  [OK] {m}: {df[col].notna().sum()} samples')
    else:
        print(f'  [--] {m}: not available')

  [OK] center_fan: 84 samples
  [OK] planar_removed_part_aware: 84 samples
  [OK] planar_largest_hole_only: 84 samples
  [OK] trimesh_fill_all: 84 samples
  [OK] advancing_front_rpa: 48 samples
  [OK] open3d_poisson: 84 samples


In [9]:
# Figure: SOTA comparison bar chart
fig_dir = cfg['paths']['figures_dir']

plot_sota_comparison(
    df, available_methods,
    os.path.join(fig_dir, 'fig_sota_comparison.pdf')
)
print('Saved fig_sota_comparison.pdf')

# Figure: Quantitative summary (ours only, Fig 4)
plot_quantitative_summary(
    df, os.path.join(fig_dir, 'fig4_quantitative_summary.pdf'),
    methods=None,
)
print('Saved fig4_quantitative_summary.pdf')

# Figure: Failure cases
plot_failure_cases(df, os.path.join(fig_dir, 'fig5_failure_cases.pdf'))
print('Saved fig5_failure_cases.pdf')

Saved fig_sota_comparison.pdf
Saved fig4_quantitative_summary.pdf
Saved fig5_failure_cases.pdf


In [7]:
# Distance metrics summary
dist_cols = [
    'chamfer_distance', 'hausdorff_distance', 'dev_mean', 'dev_max'
]

print(f'{"Method":<25} {"Chamfer":>10} {"Hausdorff":>10} {"Dev Mean":>10} {"Dev Max":>10}')
print('-' * 70)
for m in available_methods:
    vals = []
    for dc in dist_cols:
        col = f'{m}/{dc}'
        vals.append(df[col].mean() if col in df.columns else float('nan'))
    label = m.replace('_', ' ')
    print(f'{label:<25} {vals[0]:>10.6f} {vals[1]:>10.6f} {vals[2]:>10.6f} {vals[3]:>10.6f}')

Method                       Chamfer  Hausdorff   Dev Mean    Dev Max
----------------------------------------------------------------------
center fan                  0.081629   0.602517   0.012364   0.057972
planar removed part aware   0.081144   0.602516   0.011972   0.053469
planar largest hole only    0.079868   0.604043   0.010535   0.051590
trimesh fill all            0.079017   0.603997   0.010238   0.038263
advancing front rpa         0.079703   0.589082   0.011478   0.050984
open3d poisson              0.097364   0.602787   0.028394   0.205366


In [8]:
# Save LaTeX tables
tables_dir = os.path.join(cfg['paths']['output_dir'], 'tables')
for name, tbl in tables.items():
    latex_path = os.path.join(tables_dir, f'{name}.tex')
    tbl.to_latex(latex_path, index=False, float_format='%.4f',
                 caption=name.replace('_', ' ').title(),
                 label=f'tab:{name}')
    print(f'Saved {latex_path}')

print('\nDone! All SOTA comparison tables and figures generated.')

Saved D:\MyJupyter\Works\3DPART\outputs\tables\table2_quantitative.tex
Saved D:\MyJupyter\Works\3DPART\outputs\tables\table3_locality.tex
Saved D:\MyJupyter\Works\3DPART\outputs\tables\table4_failures.tex
Saved D:\MyJupyter\Works\3DPART\outputs\tables\table5_target_selection.tex
Saved D:\MyJupyter\Works\3DPART\outputs\tables\table6_sota_comparison.tex
Saved D:\MyJupyter\Works\3DPART\outputs\tables\table7_distance.tex

Done! All SOTA comparison tables and figures generated.
